# Credit Train — Feature Selection

**Upstream:** `02_credit_feature_engineering.ipynb`
**Input:** `features/credit_train_features.csv`
**Output:** Ranked feature lists and the **Standard (25)** set

---

### Purpose

Reduce the engineered matrix to a stable, non-redundant subset for modelling. Selection combines **statistical association**, **multicollinearity control**, and **model-based importance** into a consensus ranking.

### Selection gates

1. Drop constant, near-constant, and ultra-sparse columns
2. Univariate ranking (Pearson correlation, mutual information)
3. Remove one feature from each highly correlated pair
4. Random Forest impurity importance
5. Consensus score → Core / Standard / Extended sets
6. Optional diagnostic cross-validation on candidate sets


## 0. Setup

Load the engineered matrix and prepare ranking utilities for consensus selection.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams["figure.figsize"] = (10, 5)

FEAT_DIR = Path("features")
DATA = FEAT_DIR / "credit_train_features.parquet"
DICT = FEAT_DIR / "feature_dictionary.csv"
LIST = FEAT_DIR / "model_feature_list.json"

assert DATA.exists(), DATA.resolve()
print("Ready.")


## 1. Load engineered features

Read the feature matrix produced by the engineering notebook and separate predictors from the target.


In [ ]:
df = pd.read_parquet(DATA)
print(f"Loaded: {df.shape[0]:,} x {df.shape[1]}")

with open(LIST) as f:
    meta = json.load(f)

feature_cols = meta["features"]
target_col = meta["target"]
id_cols = meta["id_cols"]

feature_cols = [c for c in feature_cols if c in df.columns]
X = df[feature_cols].copy()
y = df[target_col].copy()

feat_dict = pd.read_csv(DICT) if DICT.exists() else pd.DataFrame()
print(f"Features: {len(feature_cols)}  |  Default rate: {y.mean()*100:.2f}%")
X.head(3)


## 2. Gate A — Missingness and variance filters

Remove columns that are constant, nearly constant, or too sparse to support stable estimates.


In [ ]:
miss_pct = X.isna().mean() * 100
miss_pct = miss_pct.sort_values(ascending=False)

print("Missingness summary:")
print(f"  Fully populated: {(miss_pct == 0).sum()}")
print(f"  0-20% missing:   {((miss_pct > 0) & (miss_pct <= 20)).sum()}")
print(f"  20-60% missing:  {((miss_pct > 20) & (miss_pct <= 60)).sum()}")
print(f"  >60% missing:    {(miss_pct > 60).sum()}")
print()
print(miss_pct.head(12).to_string())


In [ ]:
def variance_report(frame):
    rows = []
    for c in frame.columns:
        s = frame[c]
        nunique = s.dropna().nunique()
        if nunique <= 1:
            rows.append({"feature": c, "nunique": nunique, "var": 0.0,
                         "prevalence": np.nan, "flag": "constant"})
            continue
        var = s.var(skipna=True)
        is_bin = nunique == 2 and set(s.dropna().unique()).issubset({0, 1, 0.0, 1.0})
        prev = float(s.mean()) if is_bin else np.nan
        flag = "ok"
        if is_bin and (prev < 0.005 or prev > 0.995):
            flag = "rare_binary"
        rows.append({"feature": c, "nunique": nunique, "var": var,
                     "prevalence": prev, "flag": flag})
    return pd.DataFrame(rows)

vrep = variance_report(X)
print(vrep["flag"].value_counts().to_string())
print()
print("Flagged features:")
print(vrep[vrep["flag"] != "ok"].to_string(index=False))


In [ ]:
MISS_THRESHOLD = 60.0
drop_missing = miss_pct[miss_pct > MISS_THRESHOLD].index.tolist()
drop_missing = [c for c in drop_missing if not str(c).endswith("_missing")]

drop_variance = vrep.loc[vrep["flag"].isin(["constant", "rare_binary"]), "feature"].tolist()

drop_gate_a = sorted(set(drop_missing + drop_variance))
keep_after_a = [c for c in feature_cols if c not in drop_gate_a]

print(f"Dropped by Gate A: {len(drop_gate_a)}")
for c in drop_gate_a:
    reason = []
    if c in drop_missing:
        reason.append(f"missing={miss_pct[c]:.1f}%")
    if c in drop_variance:
        reason.append(vrep.loc[vrep["feature"] == c, "flag"].iloc[0])
    print(f"  - {c}: {', '.join(reason)}")
print(f"Remaining: {len(keep_after_a)}")

X_a = X[keep_after_a].copy()


## 3. Gate B — Univariate association with Default

Methods:
- **Pearson |r|** (linear association)
- **Mutual information** (non-linear, model-free)


In [ ]:
imputer = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imputer.fit_transform(X_a), columns=X_a.columns, index=X_a.index)

pearson = X_imp.apply(lambda col: col.corr(y))
pearson_abs = pearson.abs().sort_values(ascending=False)

mi = mutual_info_classif(X_imp, y, discrete_features=False, random_state=42)
mi_s = pd.Series(mi, index=X_imp.columns).sort_values(ascending=False)

univ = pd.DataFrame({
    "pearson_r": pearson,
    "pearson_abs": pearson_abs,
    "mutual_info": mi_s,
})
univ["pearson_rank"] = univ["pearson_abs"].rank(ascending=False)
univ["mi_rank"] = univ["mutual_info"].rank(ascending=False)
univ["univ_rank_avg"] = (univ["pearson_rank"] + univ["mi_rank"]) / 2
univ = univ.sort_values("univ_rank_avg")

print("Top 20 by average univariate rank:")
print(univ.head(20)[["pearson_r", "pearson_abs", "mutual_info", "univ_rank_avg"]].to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

top_p = pearson_abs.head(15)
colors_p = ["#e74c3c" if pearson[c] > 0 else "#2ecc71" for c in top_p.index]
axes[0].barh(top_p.index[::-1], top_p.values[::-1], color=colors_p[::-1])
axes[0].set_xlabel("|Pearson r|")
axes[0].set_title("Top 15 by |Pearson r| with Default")

top_m = mi_s.head(15)
axes[1].barh(top_m.index[::-1], top_m.values[::-1], color="#3498db")
axes[1].set_xlabel("Mutual information")
axes[1].set_title("Top 15 by Mutual Information")

plt.tight_layout()
plt.show()


## 4. Gate C — Multicollinearity and redundancy

Within highly correlated pairs, keep the feature with the stronger univariate association to Default.


In [ ]:
CORR_THRESHOLD = 0.90
corr = X_imp.corr().abs()

pairs = []
cols = corr.columns.tolist()
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        r = corr.iloc[i, j]
        if r >= CORR_THRESHOLD:
            a, b = cols[i], cols[j]
            score_a = pearson_abs.get(a, 0)
            score_b = pearson_abs.get(b, 0)
            drop = b if score_a >= score_b else a
            keep = a if drop == b else b
            pairs.append({
                "feature_a": a, "feature_b": b, "abs_r": r,
                "keep": keep, "drop": drop,
                "score_keep": max(score_a, score_b),
                "score_drop": min(score_a, score_b),
            })

redundancy = pd.DataFrame(pairs).sort_values("abs_r", ascending=False) if pairs else pd.DataFrame()
print(f"Highly correlated pairs (|r| >= {CORR_THRESHOLD}): {len(redundancy)}")
if len(redundancy):
    print(redundancy.head(20).to_string(index=False))
else:
    print("None found.")


In [ ]:
drop_redundant = sorted(redundancy["drop"].unique().tolist()) if len(redundancy) else []
drop_redundant = [c for c in drop_redundant if c in keep_after_a]
keep_after_c = [c for c in keep_after_a if c not in drop_redundant]

print(f"Dropped as redundant: {len(drop_redundant)}")
for c in drop_redundant:
    print(f"  - {c}")
print(f"Remaining: {len(keep_after_c)}")

X_c = X_imp[keep_after_c].copy()


In [ ]:
top_for_heat = pearson_abs.reindex(keep_after_c).sort_values(ascending=False).head(18).index.tolist()
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    X_c[top_for_heat].corr(), annot=True, fmt=".2f", cmap="RdBu_r",
    center=0, vmin=-1, vmax=1, ax=ax, square=True, annot_kws={"size": 7},
)
ax.set_title("Correlation among top remaining features")
plt.tight_layout()
plt.show()


## 5. Gate D — Model-based importance

Fit a Random Forest on the filtered set and rank features by impurity-based importance.


In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=30,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_c, y)

rf_imp = pd.Series(rf.feature_importances_, index=X_c.columns).sort_values(ascending=False)
print("Top 20 Random Forest importances:")
print(rf_imp.head(20).to_string())

fig, ax = plt.subplots(figsize=(9, 6))
top_rf = rf_imp.head(20)
ax.barh(top_rf.index[::-1], top_rf.values[::-1], color="#8e44ad")
ax.set_xlabel("Impurity importance")
ax.set_title("Top 20 Random Forest feature importances")
plt.tight_layout()
plt.show()


## 6. Consensus ranking

Average rank positions across correlation, mutual information, and forest importance to form a consensus score.


In [ ]:
consensus = pd.DataFrame(index=keep_after_c)
consensus["pearson_r"] = pearson.reindex(keep_after_c)
consensus["pearson_abs"] = pearson_abs.reindex(keep_after_c)
consensus["mutual_info"] = mi_s.reindex(keep_after_c)
consensus["rf_importance"] = rf_imp.reindex(keep_after_c)

consensus["rank_pearson"] = consensus["pearson_abs"].rank(ascending=False)
consensus["rank_mi"] = consensus["mutual_info"].rank(ascending=False)
consensus["rank_rf"] = consensus["rf_importance"].rank(ascending=False)
consensus["rank_avg"] = consensus[["rank_pearson", "rank_mi", "rank_rf"]].mean(axis=1)
consensus["rank_std"] = consensus[["rank_pearson", "rank_mi", "rank_rf"]].std(axis=1)

consensus = consensus.sort_values("rank_avg")
consensus["consensus_rank"] = range(1, len(consensus) + 1)

if len(feat_dict):
    fam = feat_dict.drop_duplicates("feature").set_index("feature")["family"]
    consensus["family"] = fam.reindex(consensus.index)

print("Consensus top 25:")
cols_show = ["consensus_rank", "pearson_r", "mutual_info", "rf_importance", "rank_avg"]
if "family" in consensus.columns:
    cols_show = ["consensus_rank", "family", "pearson_r", "mutual_info", "rf_importance", "rank_avg"]
print(consensus[cols_show].head(25).to_string())


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
top_c = consensus.head(20)
ax.barh(
    top_c.index[::-1],
    (1 / top_c["rank_avg"]).values[::-1],
    color="#16a085",
)
ax.set_xlabel("1 / average rank  (higher = more consensus importance)")
ax.set_title("Top 20 features by consensus rank")
plt.tight_layout()
plt.show()


## 7. Recommended feature sets

| Tier | Size | Use case |
|------|------|----------|
| **Core** | top 15 | Interpretable scorecard / logistic regression |
| **Standard** | top 25 | Gradient boosting / RF production model |
| **Extended** | top 40 | Feature exploration / ensemble stacking |


In [ ]:
CORE_K, STANDARD_K, EXTENDED_K = 15, 25, 40

core = consensus.head(CORE_K).index.tolist()
standard = consensus.head(STANDARD_K).index.tolist()
extended = consensus.head(min(EXTENDED_K, len(consensus))).index.tolist()

print(f"Core ({len(core)}):")
for i, c in enumerate(core, 1):
    print(f"  {i:2d}. {c}")
print(f"\nStandard ({len(standard)}):")
for i, c in enumerate(standard, 1):
    print(f"  {i:2d}. {c}")
print(f"\nExtended ({len(extended)}):")
for i, c in enumerate(extended, 1):
    print(f"  {i:2d}. {c}")


## 8. Quick predictive sanity check

Optional cross-validated AUC on candidate sets to confirm that smaller tiers retain most of the signal.


In [ ]:
def eval_feature_set(cols, label):
    X_sub = X_c[cols]
    results = {}
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    pipe_lr = Pipeline([
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(max_iter=500, class_weight="balanced", random_state=42)),
    ])
    scores_lr = cross_val_score(pipe_lr, X_sub, y, cv=cv, scoring="roc_auc", n_jobs=-1)
    results["lr_auc"] = float(scores_lr.mean())
    results["lr_std"] = float(scores_lr.std())

    pipe_rf = Pipeline([
        ("clf", RandomForestClassifier(
            n_estimators=100, max_depth=8, min_samples_leaf=40,
            class_weight="balanced_subsample", n_jobs=-1, random_state=42,
        )),
    ])
    scores_rf = cross_val_score(pipe_rf, X_sub, y, cv=cv, scoring="roc_auc", n_jobs=-1)
    results["rf_auc"] = float(scores_rf.mean())
    results["rf_std"] = float(scores_rf.std())
    results["n_features"] = len(cols)
    results["set"] = label
    return results

diag = []
for cols, label in [
    (core, "core"),
    (standard, "standard"),
    (extended, "extended"),
    (keep_after_c, "all_filtered"),
]:
    print(f"Evaluating: {label} ({len(cols)} features)...")
    diag.append(eval_feature_set(cols, label))

diag_df = pd.DataFrame(diag)[["set", "n_features", "lr_auc", "lr_std", "rf_auc", "rf_std"]]
print()
print(diag_df.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(diag_df))
w = 0.35
ax.bar(x - w/2, diag_df["lr_auc"], w, label="Logistic Regression", color="#3498db",
       yerr=diag_df["lr_std"], capsize=4)
ax.bar(x + w/2, diag_df["rf_auc"], w, label="Random Forest", color="#8e44ad",
       yerr=diag_df["rf_std"], capsize=4)
ax.set_xticks(x)
ax.set_xticklabels([f"{r['set']}\n(n={int(r['n_features'])})" for _, r in diag_df.iterrows()])
ax.set_ylabel("CV ROC-AUC")
ax.set_title("Feature-set diagnostic (3-fold CV ROC-AUC)")
ax.legend()
ax.set_ylim(0.5, max(diag_df["lr_auc"].max(), diag_df["rf_auc"].max()) + 0.05)
plt.tight_layout()
plt.show()


## 9. Persist selection artifacts

Save ranking tables, selected feature JSON lists, and diagnostic plots for the modelling notebook.


In [ ]:
consensus_out = consensus.reset_index().rename(columns={"index": "feature"})
consensus_path = FEAT_DIR / "feature_selection_ranking.csv"
consensus_out.to_csv(consensus_path, index=False)

selection = {
    "core": core,
    "standard": standard,
    "extended": extended,
    "all_filtered": keep_after_c,
    "dropped_gate_a": drop_gate_a,
    "dropped_redundant": drop_redundant,
    "gates": {
        "missing_threshold_pct": MISS_THRESHOLD,
        "corr_threshold": CORR_THRESHOLD,
        "rare_binary_prevalence": [0.005, 0.995],
    },
    "diagnostic_cv_auc": diag_df.to_dict(orient="records"),
}
selection_path = FEAT_DIR / "selected_feature_sets.json"
with open(selection_path, "w") as f:
    json.dump(selection, f, indent=2)

default_list_path = FEAT_DIR / "selected_features_standard.json"
with open(default_list_path, "w") as f:
    json.dump({"features": standard, "target": "Default", "tier": "standard"}, f, indent=2)

print("Saved:")
print(f"  {consensus_path}")
print(f"  {selection_path}")
print(f"  {default_list_path}")


## 10. Summary & recommendations

### Pipeline gates applied

1. **Gate A** — removed constant / rare binary / ultra-sparse features
2. **Gate B** — ranked by Pearson + mutual information
3. **Gate C** — removed one side of |r| ≥ 0.90 pairs (kept stronger target link)
4. **Gate D** — Random Forest impurity importance
5. **Consensus** — average of the three ranks

### How to use the tiers

- **Core (15):** scorecards, regulatory logistic models, quick baselines
- **Standard (25):** recommended default for GBM / RF production candidates
- **Extended (40):** ablation studies, stacking meta-features

### Modelling notes

- Always **impute after the train/validation split** (median shown here is only for ranking diagnostics)
- Prefer **Standard** tier unless latency or interpretability constraints force Core
- Re-run this notebook if feature engineering changes — ranks will shift
- Missingness flags that survive selection should stay with their parent features at inference time


In [ ]:
print("=" * 60)
print("FEATURE SELECTION COMPLETE")
print("=" * 60)
print(f"Input features:     {len(feature_cols)}")
print(f"After Gate A:       {len(keep_after_a)}")
print(f"After Gate C:       {len(keep_after_c)}")
print(f"Core / Standard / Extended: {len(core)} / {len(standard)} / {len(extended)}")
print(f"Best diagnostic RF AUC (standard): "
      f"{diag_df.loc[diag_df['set']=='standard', 'rf_auc'].values[0]:.4f}")
print(f"Top consensus feature: {consensus.index[0]}")
print("=" * 60)
